In [ ]:
NOTEBOOK_TITLE = '610: DueCare Submission Walkthrough'
from IPython.display import HTML, display
display(HTML(
    '''<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);color:white;padding:20px 24px;border-radius:8px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif">'''
    '''<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;opacity:0.8;text-transform:uppercase">DueCare - Gemma 4 Good Hackathon</div>'''
    f'''<div style="font-size:24px;font-weight:700;margin:4px 0 0 0">{NOTEBOOK_TITLE}</div>'''
    '''<div style="font-size:13px;opacity:0.92;margin-top:4px">Fine-tuned Gemma 4 as an on-device safety judge. Privacy is non-negotiable.</div></div>'''
))

_P = {"primary":"#4c78a8","success":"#10b981","info":"#3b82f6","warning":"#f59e0b","muted":"#6b7280",
      "bg_primary":"#eff6ff","bg_success":"#ecfdf5","bg_info":"#eff6ff","bg_warning":"#fffbeb"}
def _card(v, l, s, k="primary"):
    c = _P[k]; bg = _P.get(f"bg_{k}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:top;width:22%;margin:4px 1%;padding:14px 16px;'
            f'background:{bg};border-left:5px solid {c};border-radius:4px;font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-size:11px;font-weight:600;color:{c};text-transform:uppercase;letter-spacing:0.04em">{l}</div>'
            f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{v}</div>'
            f'<div style="font-size:12px;color:{_P["muted"]};margin-top:2px">{s}</div></div>')

cards = [
    _card('on-device', 'runtime', 'privacy-preserving', 'success'),
    _card('Gemma 4', 'model family', 'E2B / E4B / 31B', 'primary'),
    _card('6-dim', 'rubric', 'consistent across suite', 'info'),
    _card('open', 'license', 'CC-BY 4.0 per comp rules', 'warning'),
]
display(HTML('<div style="margin:6px 0">' + ''.join(cards) + '</div>'))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 610: DueCare Submission Walkthrough

**This is the judge-facing capstone notebook.** [600 Results Dashboard](https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard) carries the measured charts. [620 Demo API Endpoint Tour](https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour) shows the deployed surface, including the NGO migration-case workflow. [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough) shows how a partner adopts the same system in a new domain. 610 stitches those proof surfaces into one submission story a reviewer can verify in minutes.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a). Gemma 4's native function calling is load-bearing in the agent swarm. Gemma 4's multimodal understanding is load-bearing in the document-analysis and case-file path. This notebook stays CPU-only by verifying the installed package, real registries, and a scripted cross-domain run rather than trying to replay GPU-heavy notebooks inline.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">The installed <code>duecare-llm</code> meta package, live registry counts, and proof surfaces already established elsewhere in the suite: <a href='https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard'>600 Results Dashboard</a>, <a href='https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour'>620 Demo API Endpoint Tour</a>, and <a href='https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough'>650 Custom Domain Walkthrough</a>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">One capstone claim cell, a reader-facing surface map, registry counts, and a scripted guardrails run across the shipped <code>trafficking</code>, <code>tax_evasion</code>, and <code>financial_crime</code> packs. The operator story now explicitly includes the NGO migration-case workflow before the final handoff to 620, 650, and 899.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached. No GPU or API keys. Reading <a href='https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard'>600 Results Dashboard</a> first is recommended because 610 is the narrative capstone that picks up after the measured proof.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 90 seconds end to end. This notebook only installs the pinned package, inspects registries, and runs a scripted cross-domain proof.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Solution Surfaces capstone. Previous: <a href='https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard'>600 Results Dashboard</a>. Next: <a href='https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour'>620 Demo API Endpoint Tour</a>. Adoption path: <a href='https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough'>650 Custom Domain Walkthrough</a>. Section close: <a href='https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion'>899 Solution Surfaces Conclusion</a>.</td></tr>
  </tbody>
</table>

### Why this notebook exists

A judge should not have to reverse-engineer the repo. 610 is the short answer to three questions: what ships, who uses it, and why the claim is credible. The suite exposes four public notebook surfaces, but five deployment shapes, because the 620 API notebook carries both the generalized NGO API and the migration-case workflow.

### Reading order

- **Previous proof surface:** [600 Results Dashboard](https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard) contains the measured charts used in the video and writeup.
- **Earlier evidence:** [010 Quickstart](https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes) proves the local install path, [200 Cross-Domain Proof](https://www.kaggle.com/code/taylorsamarel/duecare-200-cross-domain-proof) proves the harness generalizes, [500 Agent Swarm Deep Dive](https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive) proves the coordinator and agents are real, and [530 Phase 3 Unsloth Fine-tune](https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune) is the improvement path that eventually feeds 600.
- **Next surfaces:** [620 Demo API Endpoint Tour](https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour) for the deployed API story and NGO case-bundle workflow, [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough) for partner adoption, and [899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion) for the section close.
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).

### What this notebook does

1. Install the meta package and verify the <code>duecare</code> namespace is intact.
2. Print the submission claim in one cell.
3. Map the four public notebook surfaces the submission actually ships.
4. Count the registries so the package shape is visible.
5. Run one scripted cross-domain proof across trafficking, tax evasion, and financial crime.
6. Close with the deployer story, named partners, and a strong handoff.


## Install and verify the meta package

In [ ]:
import duecare.core
import duecare.cli

print(f'Installed: duecare-llm {duecare.core.__version__}')
print('Meta package import path verified: duecare.cli')


## The submission in one cell

In [ ]:
CLAIM = '''
DueCare is a private, agentic safety harness for LLMs.

  install -> select domain pack -> run tasks -> inspect failures ->
  fine-tune -> validate -> publish

One package. One CLI. One registry-driven architecture.

Gemma 4 is the first full benchmark and deployment story.
The same harness also runs on tax evasion and financial
crime with zero code changes.
'''
print(CLAIM)


## What ships

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 19%;">Surface</th>
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Primary user</th>
      <th style="padding: 6px 10px; text-align: left; width: 23%;">Notebook</th>
      <th style="padding: 6px 10px; text-align: left; width: 36%;">What it proves</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Private local install</b></td><td style="padding: 6px 10px;">NGO staff, judges, regulators</td><td style="padding: 6px 10px;"><a href='https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes'>010 Quickstart</a></td><td style="padding: 6px 10px;"><code>pip install duecare-llm</code> works on a laptop and the namespace resolves cleanly.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Measured proof surface</b></td><td style="padding: 6px 10px;">Judges, writeup, video viewers</td><td style="padding: 6px 10px;"><a href='https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard'>600 Results Dashboard</a></td><td style="padding: 6px 10px;">The baseline and improvement story is visible in charts instead of prose.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Deployment API</b></td><td style="padding: 6px 10px;">NGO engineers, product teams</td><td style="padding: 6px 10px;"><a href='https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour'>620 Demo API Endpoint Tour</a></td><td style="padding: 6px 10px;">The web app and REST contract are concrete, inspectable, and now include a multi-document migration-case workflow with timelines, grounding, and complaint drafts.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Custom domain adoption</b></td><td style="padding: 6px 10px;">Partner researchers and NGOs</td><td style="padding: 6px 10px;"><a href='https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough'>650 Custom Domain Walkthrough</a></td><td style="padding: 6px 10px;">A new domain pack can be added without Python changes, which is the reusability story judges will test.</td></tr>
  </tbody>
</table>

The technical depth behind these surfaces lives in [500 Agent Swarm Deep Dive](https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive) and [530 Phase 3 Unsloth Fine-tune](https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune). 610 is the product-facing stitch, not a substitute for those deeper proofs. Read the surface count as four public notebooks and five deployment shapes: 620 covers both the generalized operator API and the multi-document migration-case path.


## Verify all 8 sub-packages imported via the `duecare` namespace

In [ ]:
import duecare.models, duecare.domains, duecare.tasks, duecare.agents, duecare.workflows, duecare.publishing, duecare.cli

modules = [
    ('duecare.core', 'Contracts + schemas + registries'),
    ('duecare.models', '8 model adapters'),
    ('duecare.domains', 'Pluggable domain packs'),
    ('duecare.tasks', '9 capability tests'),
    ('duecare.agents', '12-agent swarm + supervisor'),
    ('duecare.workflows', 'YAML DAG runner'),
    ('duecare.publishing', 'HF Hub + Kaggle + reports'),
    ('duecare.cli', '`duecare` command-line entry point'),
]
for mod_name, desc in modules:
    print(f'  {mod_name:<25} - {desc}')


## Count what the harness knows about

In [ ]:
from duecare.models import model_registry
from duecare.tasks import task_registry
from duecare.agents import agent_registry
from duecare.domains import register_discovered, domain_registry

register_discovered()

print(f'Model adapters:    {len(model_registry)}')
print(f'Capability tests:  {len(task_registry)}')
print(f'Agents in swarm:   {len(agent_registry)}')
print(f'Domain packs:      {len(domain_registry)}')


## Cross-domain proof (the killer demo)

In [ ]:
from duecare.core import ChatMessage, Capability, GenerationResult, Embedding, ModelHealth, TaskConfig
from duecare.domains import domain_registry, load_domain_pack
from duecare.tasks import task_registry


class ScriptedModel:
    id = 'scripted:submission'
    display_name = 'Scripted Submission Model'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _r = (
        'I cannot help with that. Under ILO C181, FATF 40 Recommendations, '
        'and OECD BEPS, this is illegal. Please contact the relevant '
        'authority (POEA, IRS, FinCEN, or your local ministry).'
    )
    def generate(self, messages, tools=None, images=None, max_tokens=1024, temperature=0.0, **kwargs):
        return GenerationResult(text=self._r, model_id=self.id)
    def embed(self, texts):
        return [Embedding(text=t, vector=[0.0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self):
        return ModelHealth(model_id=self.id, healthy=True)

m = ScriptedModel()
task = task_registry.get('guardrails')
validated_domains = []

print(f'{"Domain":<20} {"mean_score":>12} {"refusal_rate":>14} {"prompts":>10}')
print('-' * 60)

for domain_id in ['trafficking', 'tax_evasion', 'financial_crime']:
    if not domain_registry.has(domain_id):
        print('{:<20} {:>12} {:>14} {:>10}'.format(domain_id, 'MISSING', 'MISSING', 'MISSING'))
        continue
    pack = load_domain_pack(domain_id)
    r = task.run(m, pack, TaskConfig())
    validated_domains.append(domain_id)
    print(
        f'{domain_id:<20} '
        f'{r.metrics["mean_score"]:>12.4f} '
        f'{r.metrics["refusal_rate"]:>14.4f} '
        f'{int(r.metrics["n_prompts"]):>10}'
    )

print()
print(f'Validated shipped packs: {len(validated_domains)} -> {validated_domains}')


## Why the claim is credible

- [600 Results Dashboard](https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard) is the measured proof surface.
- [010 Quickstart](https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes) is the local reproducibility surface.
- [620 Demo API Endpoint Tour](https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour) is the operator surface, including the NGO migration-case intake.
- [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough) is the adoption surface.
- [530 Phase 3 Unsloth Fine-tune](https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune) is the improvement path that eventually feeds 600.

Named deployers are not hypothetical: Polaris Project, IJM, ECPAT, POEA, BP2MI, and HRD Nepal are exactly the kind of organizations this package is built for.

**Privacy is non-negotiable. So the lab runs on your machine.**


## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 34%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 66%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Install cell cannot find wheels or PyPI is blocked</td><td style="padding: 6px 10px;">Attach <code>taylorsamarel/duecare-llm-wheels</code>, keep internet enabled, and rerun the first code cell. The hardener already falls back from PyPI to attached wheels.</td></tr>
    <tr><td style="padding: 6px 10px;">Registry counts are unexpectedly low</td><td style="padding: 6px 10px;">Rerun the install cell, then rerun the namespace and registry cells in order. A stale environment usually means the pinned package was not installed cleanly.</td></tr>
    <tr><td style="padding: 6px 10px;">Cross-domain proof shows MISSING for one of the shipped packs</td><td style="padding: 6px 10px;">Confirm <code>register_discovered()</code> ran in the prior cell and that the domains wheel is present. 610 expects the bundled <code>trafficking</code>, <code>tax_evasion</code>, and <code>financial_crime</code> packs.</td></tr>
    <tr><td style="padding: 6px 10px;">You want the measured before and after story, not the narrative stitch</td><td style="padding: 6px 10px;">Open <a href='https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard'>600 Results Dashboard</a>. 610 summarizes the product surface; 600 contains the charts judges will quote.</td></tr>
  </tbody>
</table>


In [ ]:
print('Submission walkthrough complete. Next: 620 Demo API Endpoint Tour https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour | 650 Custom Domain Walkthrough https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough | 899 Solution Surfaces Conclusion https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion')
